In [1]:
!git clone https://github.com/BabarAli13/ib-dta.git /kaggle/working/ib-dta
%cd /kaggle/working/ib-dta
!pip install -q rdkit pyyaml

import glob, shutil, os
os.makedirs('/kaggle/working/results/davis', exist_ok=True)
for src in glob.glob('/kaggle/input/**/results/davis/*', recursive=True):
    if not os.path.isdir(src):
        continue
    dst = '/kaggle/working/results/davis/' + os.path.basename(src)
    if not os.path.exists(dst):
        shutil.copytree(src, dst)
n = len(glob.glob('/kaggle/working/results/davis/*'))
print(f'{n} runs carried over')
assert n >= 30, f'EXPECTED >=30 CARRIED OVER, GOT {n} - fix the input first'

!python download_data.py --datasets davis
!python splits.py --csv datasets/davis/full.csv --out datasets/davis
!python benchmark_split.py --datasets davis
!python preflight.py

Cloning into '/kaggle/working/ib-dta'...
remote: Enumerating objects: 133, done.
remote: Counting objects: 100% (133/133), done.
remote: Compressing objects: 100% (121/121), done.
remote: Total 133 (delta 44), reused 20 (delta 8), pack-reused 0 (from 0)
Receiving objects: 100% (133/133), 811.22 KiB | 9.11 MiB/s, done.
Resolving deltas: 100% (44/44), done.
/kaggle/working/ib-dta
35 runs carried over

=== davis (regression) ===
  68 ligands x 442 proteins -> matrix (68, 442)
  converted Kd(nM) -> pKd   range 5.00 .. 10.80
  wrote 30,056 pairs -> ./datasets/davis/full.csv
  68 unique drugs, 379 unique targets

Done. Next:
  python splits.py --csv ./datasets/davis/full.csv --out ./datasets/davis
  python train_ib.py --cfg configs/IBDTA.yaml --data davis --split cold_target
=== data integrity ===
  rows                     30,056
  unique_pairs             25,772
  duplicate_groups         1,224
  conflicting_groups       546
  rows_in_conflict         2,894
  pct_rows_affected        9.628

In [2]:
!CUDA_VISIBLE_DEVICES=0 python run_davis.py --only benchmark --hours 10 --out /kaggle/working/results > gpu0.log 2>&1 & CUDA_VISIBLE_DEVICES=1 python run_davis.py --only frontier --hours 10 --out /kaggle/working/results > gpu1.log 2>&1 & wait

!python collect_results.py --out /kaggle/working/results
!python run_davis.py --list --out /kaggle/working/results
!tail -20 gpu0.log
!tail -20 gpu1.log

  40 run(s) -> results/davis_summary.csv

  split regimes (mean +/- sd over seeds)
                      mse                        ci                
                     mean       std count      mean       std count
split                                                              
benchmark/fold0  0.354002       NaN     1  0.838679       NaN     1
benchmark/fold1  0.364154       NaN     1  0.840246       NaN     1
benchmark/fold2  0.347114       NaN     1  0.847259       NaN     1
benchmark/fold3  0.366775       NaN     1  0.841783       NaN     1
benchmark/fold4  0.377540       NaN     1  0.834004       NaN     1
cold_drug        0.756267  0.165240     5  0.643488  0.041749     5
cold_target      0.484587  0.042923     5  0.763995  0.014422     5
random           0.339469  0.022318     3  0.846446  0.010237     3
scaffold         1.139895       NaN     1  0.597343       NaN     1 

  wrote results/table_splits.tex
  wrote results/frontier_measured.csv

  frontier (mean +/- sd ove